# Sequentially build database
We will use the canonical peptides and the C-terminal isoform peptides as background and then sequentially add the internal and N-terminal peptides. Sequentially means that we will have a new mapping excluding all the C-terminal isoforms and from that mapping we'll go through the isoforms one by one and add them if they are unique to what has been added to the databse up to that point.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import ast

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_Isoform_Databse"):
    print("WARNING: The working directory is not set to the '02_02_Isoform_Databse'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_02_Isoform_Databse' folder (\"{cwd}\").")

In [ ]:
# Data directories
digestion_output_dir = "data/digestion_output"
fasta_dir = "../01_UniProt/data/raw_data/fasta"
mapping_dir = "../01_UniProt/data/mapping/"

## Read in dataframes

In [ ]:
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

digestion_Trypsin_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_Trypsin_filtered.csv'))
digestion_TrypsinP_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_TrypsinP_filtered.csv'))

df_c_terminal_final = pd.read_csv("data/df_c_terminal.csv")

# Find unique peptides for internal and N-terminal isoforms

In [ ]:
def find_sequential_markers_internal(digestion_df, mapping_df, all_isoform_ids, c_term_df):
    """
    Priority-based sequential identification.
    c_term_df: The dataframe of already-calculated C-terminal unique peptides.
    """
    # 1. Background: All non-isoform peptides
    background_proteome = digestion_df[~digestion_df['ID'].isin(all_isoform_ids)]
    global_forbidden = set(background_proteome['seq'].unique())

    # 2. SEED: Add all peptides from the C-terminal isoforms you already decided to keep
    # This ensures internal isoforms don't claim peptides shared with your C-term unique tails
    c_term_ids = set(c_term_df['Isoform_ID'])
    c_term_peptides = digestion_df[digestion_df['ID'].isin(c_term_ids)]
    global_forbidden.update(c_term_peptides['seq'].unique())

    # Pre-process peptides for quick lookup
    pep_dict = digestion_df.groupby('ID')['seq'].apply(set).to_dict()
    final_results = []

    for _, row in mapping_df.iterrows():
        canonical_id = row['Entry']
        isoform_list = row['Isoforms']
        if isinstance(isoform_list, str):
            isoform_list = ast.literal_eval(isoform_list)
        
        # Start the local forbidden set with the global forbidden + the specific parent
        local_forbidden = global_forbidden.copy()
        if canonical_id in pep_dict:
            local_forbidden.update(pep_dict[canonical_id])
        
        # 4. Sequential Loop (Filtering for ONLY internal isoforms)
        # We assume c_term_ids have been handled, so we only look for markers for the rest
        internal_isoforms = [i for i in isoform_list if i not in c_term_ids]
        
        for idx, iso_id in enumerate(internal_isoforms):
            current_iso_peptides = pep_dict.get(iso_id, set())
            unique_to_round = current_iso_peptides - local_forbidden
            
            if unique_to_round:
                best_peptide = max(unique_to_round, key=len)
                status = "Unique Marker Found"
            else:
                best_peptide = None
                status = "Filtered (Shared)"
                
            final_results.append({
                "Isoform_ID": iso_id,
                "Unique_Peptide": best_peptide,
                "Status": status,
                "Type": "Internal"
            })
            
            # Update local forbidden so the next internal isoform is distinct from this one
            local_forbidden.update(current_iso_peptides)

    return pd.DataFrame(final_results)

# --- EXECUTION ---

# Define the set of all isoform IDs once
all_isoform_ids = set(iso_canonical_mapping_flat['Isoform_ID'].unique())

# Generate the two dataframes
sequential_results_Try = find_sequential_markers_internal(digestion_Trypsin_filtered, iso_canonical_mapping, all_isoform_ids, df_c_terminal_final)
sequential_results_TryP = find_sequential_markers_internal(digestion_TrypsinP_filtered, iso_canonical_mapping, all_isoform_ids, df_c_terminal_final)

In [ ]:
sequential_results_Try

In [ ]:
print(f"--- Summary Statistics Try ---")
print(sequential_results_Try['Status'].value_counts())

In [ ]:
print(f"--- Summary Statistics TryP ---")
print(sequential_results_TryP['Status'].value_counts())

## Build databse fasta file
Put C-terminal isoforms peptides (only C-terminus) and internal/N-terminal isoform peptides (only tryptic peptide) into a fasta file -> isoform database

In [ ]:
# Trypsin results

final_records = []

# C-terminal Isoforms 
for _, row in df_c_terminal_final.iterrows():
    iso_id = row['Isoform_ID']
    pep_seq = row['Peptide']
    rec = SeqRecord(Seq(pep_seq), id=f"sp|{iso_id}|CTERM_VAR", description="")
    final_records.append(rec)

# Add Internal/N-terminal Isoforms
for _, row in sequential_results_Try.iterrows():
    if row['Status'] == "Unique Marker Found":
        pep_seq = row['Unique_Peptide']
        iso_id = row['Isoform_ID']
        rec = SeqRecord(Seq(pep_seq), id=f"sp|{iso_id}|INTERNAL_VAR", description="")
        final_records.append(rec)

# Write to file
with open("Try_Isoform_Database_Final.fasta", "w") as output_handle:
    SeqIO.write(final_records, output_handle, "fasta")

print(f"FASTA created with {len(final_records)} entries.")

In [ ]:
# Trypsin_P results

final_records = []

# C-terminal Isoforms 
for _, row in df_c_terminal_final.iterrows():
    iso_id = row['Isoform_ID']
    pep_seq = row['Peptide']
    rec = SeqRecord(Seq(pep_seq), id=f"sp|{iso_id}|CTERM_VAR", description="")
    final_records.append(rec)

# Add Internal/N-terminal Isoforms
for _, row in sequential_results_TryP.iterrows():
    if row['Status'] == "Unique Marker Found":
        pep_seq = row['Unique_Peptide']
        iso_id = row['Isoform_ID']
        rec = SeqRecord(Seq(pep_seq), id=f"sp|{iso_id}|INTERNAL_VAR", description="")
        final_records.append(rec)

# Write to file
with open("TryP_Isoform_Database_Final.fasta", "w") as output_handle:
    SeqIO.write(final_records, output_handle, "fasta")

print(f"FASTA created with {len(final_records)} entries.")